# Project 1 ANSC 4040 Practice Test

This is a notebook to test the saved model to be run with practice test data. I will use same mean and SD as original data set, applied to new st, then make chart of animal id and MildID

In [4]:
from pathlib import Path
import joblib
import pandas as pd

In [5]:
#Load PracticeTest.csv dataset
practice_df = pd.read_csv(Path.cwd() / "PracticeTest.csv")

# Remove duplicate rows using the same preprocessing steps used to train the model.
practice_df = practice_df.drop_duplicates().copy()

import numpy as np

# Load the original training dataset used to establish the thresholds
raw_data = pd.read_csv("raw_data.csv")


# Numerical features used for outlier filtering
features_to_filter = [
    "Avgmilkflow",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec"
]

# Calculate mean and standard deviation from ORIGINAL data
raw_means = raw_data[features_to_filter].mean()
raw_stds = raw_data[features_to_filter].std()

# Calculate z-scores for the NEW dataset using original statistics
z_scores = (practice_df[features_to_filter] - raw_means) / raw_stds

# Keep rows where every selected feature has |z-score| <= 3
outlier_mask = (z_scores.abs() <= 3).all(axis=1)

# Filter the new dataset
practice_clean = practice_df.loc[outlier_mask].copy()

# Report results
print("Original new dataset rows:", len(practice_df))
print("Rows removed:", (~outlier_mask).sum())
print("Rows retained:", len(practice_clean))
print("Percentage retained:", f"{len(practice_clean) / len(practice_df) * 100:.2f}%")


Original new dataset rows: 1701003
Rows removed: 10954
Rows retained: 1690049
Percentage retained: 99.36%


In [11]:
# Create the same date-based features used to train the model.
event_date = pd.to_datetime(practice_clean["EventDate"])
practice_clean["EventYear"] = event_date.dt.year
practice_clean["EventMonth"] = event_date.dt.month
practice_clean["EventDay"] = event_date.dt.day
practice_clean["EventDayOfWeek"] = event_date.dt.dayofweek

#make AnimalID target a string for consistency with training data
practice_clean["AnimalId"] = practice_clean["AnimalId"].astype(str)

feature_cols = [
    "Avgmilkflow",
    "EventYear",
    "EventMonth",
    "EventDay",
    "EventDayOfWeek",
    "Flow30_60Session",
    "YieldFirst2Min_Session",
    "YieldSession",
    "DurationSession_sec",
    "milking",
]
practice_X = practice_clean[feature_cols].astype("float32")

print(f"Loaded {len(practice_df)} practice rows.")

Loaded 1701003 practice rows.


In [12]:
# Load the saved model
model_path = "final_rf.joblib"
final_rf = joblib.load(model_path)

In [13]:
# Make Animal ID predictions on the cleaned practice dataset
practice_predictions = practice_clean[["MilkingId"]].copy()
practice_predictions["PredictedAnimalId"] = final_rf.predict(practice_X)

display(practice_predictions.head())

,MilkingId,PredictedAnimalId
0,8,-2.6933594875752315e+18
1,17,-4.598842436109611e+18
2,19,4.421919998218574e+18
3,27,-8.424753651391543e+18
4,37,3.185786146852392e+18


In [14]:
# Assign predicted AnimalId to MilkingId and save a CSV of predictions.
practice_predictions = practice_clean[["MilkingId"]].copy()
practice_predictions["AnimalId"] = final_rf.predict(practice_X)

# Keep a clean output order
practice_predictions = practice_predictions[["MilkingId", "AnimalId"]]

output_path = Path.cwd() / "practice_predictions.csv"
practice_predictions.to_csv(output_path, index=False)

display(practice_predictions.head())
print(f"Saved {len(practice_predictions)} predictions to {output_path}")

,MilkingId,AnimalId
0,8,-2.6933594875752315e+18
1,17,-4.598842436109611e+18
2,19,4.421919998218574e+18
3,27,-8.424753651391543e+18
4,37,3.185786146852392e+18


Saved 1690049 predictions to /Users/katrinapayton/Desktop/ANSC 4040/ansci-4040-fall-2026/files/project1-ansci-4040/practice_predictions.csv
